<a href="https://colab.research.google.com/github/valeriesutanta/pcol3911/blob/main/Copy_of_1_Get_OP_Data_Generate_FingerPrints.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a model to predict organophosphate binding
Here we use values for the organophosphates from Veselinović et al [1].
As you know OPs are important in agriculture having similar effects to the carbamates we will investigate in the LD50 practical class. They are also important chemical weapons including sarin, tabun, VX, and Novichok.

I copied the supplementary table into a CSV and put it on Github.
Once we load it into pandas and look at the table you will see it has a number of columns. We are only interested in the SMILES column and the Expr column.

We will attempt to find a relationship between the chemical properties of the molecules represted by SMILES codes and the Expr values which measure binding.

In the paper they explain: "Human bimolecular rate constants at 25°C given in a $log_{10}$ scale were used as the appropriate endpoint, Ac(exp)." This refers to the $log_{10}$ of the biomolecular rate constant $k_{1}$. $K_{1}$ represents how quickly the organophosphate compounds bind to acetylcholinesterase (AChE) at 25°C.

To understand how these compounds interact with AChE, how quickly and effectively they bind to the enzyme is measured and described using rate constants. The key rate constants are:

$k_{1}$: How fast the OP binds to AChE.

$k_{-1}$: How fast the OP-AChE complex breaks apart.

$k_{2}$: How fast the OP-AChE complex changes into a more stable form.

The full equation is:
AB + E ⟷[$k_1$][$k_{-1}$] E-AB →[$k_2$] EA (+B) →[$k_3$] E + A (→[$k_4$]$E_{age}$)



Where **AB** is the free OP, **E** is free AChE, **E-AB** is bound enzyme, **EA** is phosphorylated enzyme (stable form) and **B** is leaving group, $k_3$ describes the production of free enzyme E and A which is the OP-without its leaving group (recovery), $k_4$ is the rate constant for the ageing process which leads to covalently bound OP onto the AChE enzyme which cannot be revearsed.

*See Rosenfeld, C. A., & Sultatos, L. G. (2006)[2]*
```
[1]Jovana B. Veselinović, Goran M. Nikolić, Nataša V. Trutić, Jelena V. Živković, Aleksandar M. Veselinović, (2015) Monte Carlo Method Based QSAR Models for Prediciting of Organophosphates Binding to Acetylcholinesterase. SAR and QSAR in Environmental Research, 26:6, 449-460
[2] Rosenfeld, C. A., & Sultatos, L. G. (2006). Concentration-dependent kinetics of acetylcholinesterase inhibition by the organophosphate paraoxon. Toxicol Sci, 90(2), 460-469
```



#BEFORE YOU START:

Save your own copy of this page in your OWN Google Drive:

Click **File** and choose **"save in drive"**

In [1]:
!pip install -q datamol
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 52.3 MB/s eta 0:00:00


In [2]:
import datamol as dm

In [3]:
#Import other useful libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [4]:
#Load data from Github
data = pd.read_csv("https://github.com/sladem-tox/Tox_data/raw/main/Organophosphate_ActivityValues.csv")


In [5]:
data.head(2)

,ID,SMILES,Expr,DCW,Calc,Expr-Calc,Set,DCW.1,Calc.1,Expr-Calc.1,Set.1,DCW.2,Calc.2,Expr-Calc.2,Set.2
0,1,CSP(=O)(c1ccccc1)c1ccccc1,2.34,39.01370,2.5431,-0.2031,Tr,30.28552,2.6612,-0.3212,ST,34.79287,2.3165,0.0235,Ca
1,2,CCSP(=O)(c1ccccc1)c1ccccc1,2.69,40.13724,2.8413,-0.1513,Va,31.29530,2.9406,-0.2506,Va,36.00961,2.6870,0.0030,Va


In [6]:
#Add a molecule column and make sure RDkt can convert all SMILES
from rdkit import Chem, DataStructs
from rdkit.Chem import PandasTools, AllChem
PandasTools.AddMoleculeColumnToFrame(data,'SMILES','Molecule')
data[["SMILES","Molecule"]].head(1)

,SMILES,Molecule
0,CSP(=O)(c1ccccc1)c1ccccc1,<rdkit.Chem.rdchem.Mol object at 0x7a43a0d10120>


In [7]:
#Check for smiles that rdkit can't convert to molecule.
#If sum = 0 then they are all OK
data.Molecule.isna().sum()

np.int64(0)

**Now we want to generate chemical descriptors**

https://docs.datamol.io/stable/tutorials/Descriptors.html


In [8]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
import warnings


In [9]:
def generate_fingerprint_column_in_df(data, radius=2, fp_length=2048):
    """
    Generates Morgan fingerprints from a pandas dataframe containing SMILES strings.

    Args:
        data (DataFrame): DataFrame with 'SMILES' column.
        radius (int, optional): Radius of the fingerprints. Defaults to 2.
        fp_length (int, optional): Length of the fingerprints. Defaults to 2048.

    Returns:
        Inserts "fps" column into dataframe containing fingerprints.
    """
    smiles_list = data['SMILES'].tolist()

    fingerprints = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=DeprecationWarning)
        fp_generator = GetMorganGenerator(radius=radius, fpSize=fp_length)
        for smiles in smiles_list:
            try:
                mol = Chem.MolFromSmiles(smiles)
                if mol is None:
                    # Handle invalid SMILES gracefully
                    fingerprints.append(np.nan)
                else:
                    fp = fp_generator.GetFingerprint(mol)
                    fingerprints.append(fp)
            except ValueError:
                # Handle invalid SMILES gracefully
                fingerprints.append(np.nan)

    data['fps'] = fingerprints


In [10]:


generate_fingerprint_column_in_df(data)
data.head(2)



,ID,SMILES,Expr,DCW,Calc,Expr-Calc,Set,DCW.1,Calc.1,Expr-Calc.1,Set.1,DCW.2,Calc.2,Expr-Calc.2,Set.2,Molecule,fps
0,1,CSP(=O)(c1ccccc1)c1ccccc1,2.34,39.01370,2.5431,-0.2031,Tr,30.28552,2.6612,-0.3212,ST,34.79287,2.3165,0.0235,Ca,<rdkit.Chem.rdchem.Mol object at 0x7a43a0d10120>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2,CCSP(=O)(c1ccccc1)c1ccccc1,2.69,40.13724,2.8413,-0.1513,Va,31.29530,2.9406,-0.2506,Va,36.00961,2.6870,0.0030,Va,<rdkit.Chem.rdchem.Mol object at 0x7a43a0d109e0>,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [11]:
def fingerprint_to_bits(fp):
    return list(map(int, fp.ToBitString()))


def create_fingerprint_dataframe(data):
    # Convert fingerprints to lists of bits
    bit_lists = data['fps'].apply(fingerprint_to_bits)

    # Create a DataFrame from the bit lists
    bit_df = pd.DataFrame(bit_lists.tolist(), index=data.index)

    # Add the "Expr" column
    bit_df['Expr'] = data['Expr']

    return bit_df


In [12]:
new_data = create_fingerprint_dataframe(data)
print(new_data)

     0  1  2  3  4  5  6  7  8  9  ...  2039  2040  2041  2042  2043  2044  \
0    0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
1    0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
2    0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
3    0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
4    0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
..  .. .. .. .. .. .. .. .. .. ..  ...   ...   ...   ...   ...   ...   ...   
273  0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
274  0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
275  0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
276  0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   
277  0  0  0  0  0  0  0  0  0  0  ...     0     0     0     0     0     0   

     2045  2046  2047  Expr  
0       0     0     0  2.34  
1  

In [14]:
#Save dataframe as csv
new_data.to_csv('organophosphate_fp.csv',index=None)

## Now you can download this new csv file and store it on your Github site for modelling.
**Click on the folder icon on the left and find the file inside the explorer window (on the left) and choose download.**